# DNN with Ensemble

### Imports & Global setting

In [ ]:
pip uninstall MarkupSafe

In [ ]:
pip install kerastuner

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
import gzip
import random
from tqdm.auto import trange
import pandas as pd
import numpy as np
import shap

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow import keras
import kerastuner as kt
print(tf.__version__)

from category_encoders import TargetEncoder

In [ ]:
# reset_seeds()함수를 아래와 같이 수정해야 함.
def reset_seeds(SEED, reset_graph_with_backend=None):
    if reset_graph_with_backend is not None:
        K = reset_graph_with_backend
        K.clear_session()
        tf.compat.v1.reset_default_graph()
        print("KERAS AND TENSORFLOW GRAPHS RESET")  # optional

    np.random.seed(SEED)
    random.seed(SEED)
    tf.compat.v1.set_random_seed(SEED)
#    os.environ['CUDA_VISIBLE_DEVICES'] = ''  # for GPU
    print("RANDOM SEEDS RESET: ", SEED)  # optional

In [ ]:
# 앙상블할 DNN 모델 수
DNN_MODELS = 5

### Data Loading

In [ ]:
# load and uncompress
with gzip.open('data_preprocessed.zip','rb') as f:
    X_train, y_train, X_test, ID_test, cat_features, num_features = pickle.load(f)

### Feature engineering for DNN

In [ ]:
# 결측값 처리: 범주형이냐 수치형이냐에 따라 다르게 처리
if len(num_features) > 0:
    imp = SimpleImputer(strategy='mean')
    X_train[num_features] = imp.fit_transform(X_train[num_features])
    X_test[num_features] = imp.transform(X_test[num_features])
if len(cat_features) > 0:  
    imp = SimpleImputer(strategy="most_frequent")
    X_train[cat_features] = imp.fit_transform(X_train[cat_features])
    X_test[cat_features] = imp.transform(X_test[cat_features])

In [ ]:
scaler = StandardScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

In [ ]:
en = TargetEncoder(cols = cat_features)
X_train = en.fit_transform(X_train,y_train)
X_test = en.transform(X_test)

In [ ]:
X_all_ohe = pd.get_dummies(pd.concat([X_train, X_test]), columns=cat_features)
# 특성(열) 이름에 JSON으로 해석될 때 문제를 일으킬 수 있는 특수문자 제거 
X_all_ohe.columns = [f"{i}_{col.replace('{', '').replace('}', '').replace(':', '').replace(',', '')}" for i, col in enumerate(X_all_ohe.columns)]

# train/test 분할
X_train = X_all_ohe.iloc[:X_train.shape[0],:].reset_index(drop=True)
X_test = X_all_ohe.iloc[X_train.shape[0]:,:].reset_index(drop=True)

In [ ]:
#
# Method: Using SHAP values 
#

# DF, based on which importance is checked
X_importance = X_test

# # Explain model predictions using shap library:
model = LGBMClassifier(random_state=0).fit(X_train, y_train)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_importance)

# Plot summary_plot as barplot:
shap.summary_plot(shap_values, X_importance, plot_type='bar')

shap_sum = np.abs(shap_values).mean(axis=1)[1,:]
importance_df = pd.DataFrame([X_importance.columns.tolist(), shap_sum.tolist()]).T
importance_df.columns = ['column_name', 'shap_importance']
importance_df = importance_df.sort_values('shap_importance', ascending=False)
importance_df

In [ ]:
# 지정된(SHAP_THRESHOLD) Shap feature 중요도 이상인 것만 선택
SHAP_THRESHOLD = 0.0
features_selected = importance_df.query('shap_importance > @SHAP_THRESHOLD').column_name.tolist()
X_train = X_train[features_selected]
X_test = X_test[features_selected]

print(X_train.shape)

### Random seed 변경을 통한 다수의 DNN 모델 생성

In [ ]:
# 예측값을 저장할 폴더 생성
folder = 'dnn_ensemble'
if not os.path.isdir(folder):
    os.mkdir(folder)

In [ ]:
loop = trange(DNN_MODELS )

for _ in loop:
    SEED = np.random.randint(1, 10000)              
    reset_seeds(SEED)
    
    # Define the NN architecture
    input = keras.Input(shape=(X_train.shape[1],))
    x = keras.layers.Dense(32, activation='relu')(input)
    x = keras.layers.Dense(16, activation='relu')(x)
    output = keras.layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(input, output)    

    # Choose the optimizer and the cost function
    model.compile(loss='binary_crossentropy', optimizer=keras.optimizers.Nadam(learning_rate=0.001), metrics=['acc', keras.metrics.AUC()])
    
    # Train the model
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)]
    hist = model.fit(X_train, y_train, validation_split=0.2, batch_size=4096, epochs=50, 
                 callbacks=callbacks, shuffle=False, verbose=0)
    print("validation accuracy = ", hist.history['val_acc'][-1])
    
    # Make submissions
    submission = pd.DataFrame({
        "ID": ID_test, 
        "STATUS": (model.predict(X_test).flatten() >= 0.5).astype(int)
    })
    t = pd.Timestamp.now()
    fname = f"{folder}/loop_submission_{t.month:02}{t.day:02}_{SEED:05}.csv"
    submission.to_csv(fname, index=False)
loop.close()    

### 생성된 다수의 DNN 모형을 앙상블

In [ ]:
nf = 0
for f in os.listdir(folder):
    ext = os.path.splitext(f)[-1]
    if ext == '.csv': 
        s = pd.read_csv(folder+"/"+f)
    else: 
        continue
    if len(s.columns) !=2:
        continue
    if nf == 0: 
        slist = s
    else: 
        slist = pd.merge(slist, s, on="ID", suffixes=('',f'_{nf}'))
    nf += 1

if nf >= 2:
    pred = 0
    for j in range(nf): pred = pred + slist.iloc[:,j+1]
    pred = pred / nf   
    pred = pred.apply(lambda x: 1 if x >= 0.5 else 0)    
    submission = pd.DataFrame({'ID': slist.ID, 'STATUS': pred})
    t = pd.Timestamp.now()
    fname = f"dnn_submission_{t.month:02}{t.day:02}_{t.hour:02}{t.minute:02}.csv"
    submission.to_csv(fname, index=False)

# End